# Bipartite Matching

## Learning Objectives

1. **Define** maximum bipartite matching and augmenting paths
2. **State** Berge's theorem and use it to certify optimality
3. **Implement** augmenting-path search for maximum matching
4. **Contrast** offline maximum matching with online greedy performance

## Bipartite Graphs and Matchings

A **bipartite graph** $G = (U \cup V, E)$ has vertices partitioned into two sets $U$ and $V$;
all edges go between $U$ and $V$ (none within a side).

A **matching** $M \subseteq E$ is a set of edges with no shared vertices.

- **Perfect matching:** every vertex is matched
- **Maximum matching:** largest possible $|M|$

**Why bipartite matching?** Models assignment problems:
- Queries (U) ↔ Advertisers (V)
- Workers ↔ Jobs
- Students ↔ Schools

## Augmenting Paths

An **augmenting path** (given matching $M$) is a path that:
- Starts and ends at *unmatched* vertices
- Alternates between edges *not in* $M$ and edges *in* $M$

**Effect of augmenting:** flip matched/unmatched along the path → $|M|$ increases by 1.

**Berge's Theorem:** A matching $M$ is maximum **if and only if** there is no augmenting path.

This gives an algorithm: repeatedly find augmenting paths until none exist.

In [ ]:
from collections import deque

def max_bipartite_matching(U, V, adj):
    """
    Maximum bipartite matching via BFS augmenting paths (Hopcroft-Karp style, simplified).
    adj: dict mapping each u in U to list of v in V it can be matched with.
    Returns: dict match_u (u->v) and match_v (v->u).
    """
    match_u = {u: None for u in U}
    match_v = {v: None for v in V}

    def bfs_augment(source):
        """Try to find augmenting path from unmatched source using BFS."""
        parent = {source: None}
        queue = deque([source])
        while queue:
            u = queue.popleft()
            for v in adj.get(u, []):
                if v not in parent:        # v not yet visited on V-side
                    parent[v] = u
                    if match_v[v] is None: # found augmenting path
                        # Trace back and augment
                        while v is not None:
                            u2 = parent[v]
                            old_v = match_u[u2]
                            match_u[u2] = v
                            match_v[v] = u2
                            v = old_v
                        return True
                    else:
                        next_u = match_v[v]
                        if next_u not in parent:
                            parent[next_u] = v
                            queue.append(next_u)
        return False

    matched = 0
    for u in U:
        if match_u[u] is None:
            if bfs_augment(u):
                matched += 1
    return match_u, match_v, matched

# Demo
U = ['q1','q2','q3','q4']
V = ['a1','a2','a3']
adj = {
    'q1': ['a1','a2'],
    'q2': ['a2'],
    'q3': ['a2','a3'],
    'q4': ['a3'],
}
match_u, match_v, size = max_bipartite_matching(U, V, adj)
print(f"Maximum matching size: {size}")
for u,v in match_u.items():
    if v: print(f"  {u} -> {v}")

## Online vs Offline Performance

In the **offline** setting we can find augmenting paths — achieving maximum matching.

In the **online** setting queries arrive one at a time:
- We must assign immediately (or leave unmatched)
- Cannot un-assign a previous advertiser
- Cannot look ahead

**Example where online greedy fails:**

```
Queries arrive in order: q1, q2
q1 eligible for: A, B
q2 eligible for: B only

Greedy assigns q1→B (valid), then q2 has no available advertiser → size 1
Optimal assigns q1→A, q2→B → size 2
```

CR = 1/2 on this instance.

In [ ]:
def greedy_online(U, V, adj_online):
    """Online greedy: assign each query to first available advertiser."""
    available = set(V)
    matching = {}
    for u in U:
        for v in adj_online.get(u,[]):
            if v in available:
                matching[u] = v
                available.remove(v)
                break
    return matching

# Adversarial example
U = [f'q{i}' for i in range(1,7)]
V = [f'a{i}' for i in range(1,6)]
# Each query eligible for one specific advertiser + advertiser a1 (shared)
adj = {
    'q1': ['a1'],
    'q2': ['a1','a2'],
    'q3': ['a1','a3'],
    'q4': ['a1','a4'],
    'q5': ['a1','a5'],
    'q6': ['a1'],
}

greedy = greedy_online(U, V, adj)
_, _, opt_size = max_bipartite_matching(U, V, adj)

print(f"Online greedy size:   {len(greedy)}")
print(f"Offline optimal size: {opt_size}")
print(f"CR on this instance:  {len(greedy)/opt_size:.2f}")

## Hall's Theorem

**Hall's Marriage Theorem:** A bipartite graph $G=(U\cup V, E)$ has a perfect matching (matching all of $U$) if and only if:

$$\forall S \subseteq U: |N(S)| \geq |S|$$

where $N(S)$ is the set of neighbors of $S$ in $V$.

**Corollary:** if the condition fails for some $S$ (a "Hall violator"), no perfect matching exists.

This gives a certificate of infeasibility.

In [ ]:
def find_hall_violator(U, V, adj):
    """Find a set S ⊆ U with |N(S)| < |S|, or return None if Hall's condition holds."""
    from itertools import combinations
    for size in range(1, len(U)+1):
        for S in combinations(U, size):
            NS = set()
            for u in S:
                NS.update(adj.get(u,[]))
            if len(NS) < len(S):
                return list(S), list(NS)
    return None

# Graph that violates Hall's condition
U2 = ['q1','q2','q3']
V2 = ['a1','a2']
adj2 = {'q1':['a1'],'q2':['a1','a2'],'q3':['a2']}

violator = find_hall_violator(U2, V2, adj2)
if violator:
    S, NS = violator
    print(f"Hall violator: S={S}, N(S)={NS}")
    print(f"|S|={len(S)} > |N(S)|={len(NS)} → no perfect matching")
else:
    print("Hall's condition satisfied — perfect matching exists")

## Summary

| Property | Value |
|----------|-------|
| Offline maximum matching | Polynomial via augmenting paths |
| Online greedy CR | 1/2 |
| Online RANKING CR | 1 − 1/e |
| Certifying no perfect matching | Hall's theorem / König's theorem |

The gap between offline (optimal) and online (1/2 or 1−1/e) reflects the cost of irrevocability.
The RANKING algorithm bridges most of this gap without any lookahead.